# SPO Triplet Extraction from Portal Files

Extract subject-predicate-object triplets from portal text files using the REBEL model.

In [2]:
import json
import re
import os
import glob
from typing import List, Dict, Any
import pandas as pd
import torch
from transformers import pipeline

In [3]:
# Set working directory to kings base folder
import os
os.chdir('/Users/admin-tascott/Documents/GitHub/kings')

# Configuration
CLOBBER = False
TEST = True

PORTAL_FILES_DIR = "data/Multipurpose_Files/portal_files"
PAGE_METADATA_PATH = "Network_Innovation_Paper/data_products/page_metadata.csv"
OUTPUT_DIR = "Network_Innovation_Paper/data_products/spo_graphs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
class REBELExtractor:
    def __init__(self, model_name: str = "Babelscape/rebel-large"):
        self.model_name = model_name
        self.pipeline = None
        
        try:
            self.pipeline = pipeline(
                'text2text-generation', 
                model=model_name, 
                tokenizer=model_name,
                device=0 if torch.cuda.is_available() else -1
            )
        except Exception as e:
            print(f"Error loading REBEL model: {e}")
    
    def chunk_text(self, text: str, max_tokens: int = 400) -> List[str]:
        """Split text into chunks that fit the model token limit"""
        if not self.pipeline:
            return [text]
        
        # Use the actual tokenizer to count tokens
        sentences = text.split('. ')
        chunks = []
        current_chunk = ""
        
        for sentence in sentences:
            test_chunk = current_chunk + sentence + ". " if current_chunk else sentence + ". "
            
            # Use actual tokenizer to count tokens
            tokens = self.pipeline.tokenizer.encode(test_chunk, add_special_tokens=True)
            
            if len(tokens) <= max_tokens:
                current_chunk = test_chunk
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                    current_chunk = sentence + ". "
                else:
                    # Single sentence is too long, split by words
                    words = sentence.split()
                    for word in words:
                        test_chunk = current_chunk + " " + word if current_chunk else word
                        tokens = self.pipeline.tokenizer.encode(test_chunk, add_special_tokens=True)
                        if len(tokens) <= max_tokens:
                            current_chunk = test_chunk
                        else:
                            if current_chunk:
                                chunks.append(current_chunk.strip())
                            current_chunk = word
        
        if current_chunk:
            chunks.append(current_chunk.strip())
        
        return chunks if chunks else [text[:1000]]  # fallback to truncated text
    
    def extract(self, text: str) -> List[Dict[str, Any]]:
        if not self.pipeline:
            return []
        
        # Split long text into chunks
        chunks = self.chunk_text(text)
        all_triplets = []
        
        for i, chunk in enumerate(chunks):
            if not chunk.strip():
                continue
                
            try:
                # Double-check token count before processing
                tokens = self.pipeline.tokenizer.encode(chunk, add_special_tokens=True)
                if len(tokens) > 1024:
                    print(f"Warning: Chunk {i+1} still too long ({len(tokens)} tokens), truncating")
                    # Truncate the chunk
                    truncated_tokens = tokens[:1000]  # Leave room for generation
                    chunk = self.pipeline.tokenizer.decode(truncated_tokens, skip_special_tokens=True)
                
                generated = self.pipeline(
                    chunk, 
                    return_tensors=True, 
                    return_text=False,
                    max_length=512
                )
                
                decoded = self.pipeline.tokenizer.batch_decode(
                    [generated[0]["generated_token_ids"]]
                )[0]
                
                chunk_triplets = self._parse_output(decoded)
                
                # Add chunk info to triplets
                for triplet in chunk_triplets:
                    triplet['chunk'] = i + 1
                    triplet['total_chunks'] = len(chunks)
                
                all_triplets.extend(chunk_triplets)
                
            except Exception as e:
                print(f"Error processing chunk {i+1}: {e}")
                continue
        
        return all_triplets
    
    def _parse_output(self, output: str) -> List[Dict[str, Any]]:
        triplets = []
        text = output.replace("<s>", "").replace("<pad>", "").replace("</s>", "").strip()
        
        current_triplet = {}
        tokens = text.split()
        
        i = 0
        while i < len(tokens):
            token = tokens[i]
            
            if token == "<triplet>":
                if current_triplet and all(k in current_triplet for k in ['head', 'type', 'tail']):
                    triplets.append({
                        'subject': current_triplet['head'],
                        'predicate': current_triplet['type'],
                        'object': current_triplet['tail'],
                        'confidence': 1.0
                    })
                current_triplet = {}
                
            elif token == "<subj>":
                i += 1
                subj_tokens = []
                while i < len(tokens) and tokens[i] not in ["<obj>", "<subj>", "<triplet>"]:
                    subj_tokens.append(tokens[i])
                    i += 1
                current_triplet['head'] = " ".join(subj_tokens).strip()
                i -= 1
                
            elif token == "<obj>":
                i += 1
                obj_tokens = []
                while i < len(tokens) and tokens[i] not in ["<subj>", "<obj>", "<triplet>"]:
                    obj_tokens.append(tokens[i])
                    i += 1
                current_triplet['tail'] = " ".join(obj_tokens).strip()
                i -= 1
                
            else:
                if not current_triplet.get('type') and token not in ["<triplet>", "<subj>", "<obj>"]:
                    current_triplet['type'] = token
            
            i += 1
        
        if current_triplet and all(k in current_triplet for k in ['head', 'type', 'tail']):
            triplets.append({
                'subject': current_triplet['head'],
                'predicate': current_triplet['type'], 
                'object': current_triplet['tail'],
                'confidence': 1.0
            })
        
        return triplets

In [5]:
# Initialize extractor
extractor = REBELExtractor()

if extractor.pipeline is not None:
    print("REBEL extractor ready")
else:
    print("Failed to initialize REBEL extractor")

Device set to use cpu


REBEL extractor ready


In [6]:
# Load page metadata
metadata_df = pd.read_csv(PAGE_METADATA_PATH)
print(f"Loaded {len(metadata_df)} rows of metadata")

# Filter for relevant pages
if 'sust_criteria' in metadata_df.columns and 'projects_mgmt_actions' in metadata_df.columns:
    filtered_df = metadata_df[
        (metadata_df['sust_criteria'] == True) | 
        (metadata_df['projects_mgmt_actions'] == True)
    ]
    print(f"Filtered to {len(filtered_df)} relevant pages")
    metadata_df = filtered_df
else:
    print("Warning: Expected filter columns not found")
    print(f"Available columns: {list(metadata_df.columns)}")

Loaded 79440 rows of metadata
Filtered to 13399 relevant pages


In [7]:
# Get files to process
txt_files = glob.glob(os.path.join(PORTAL_FILES_DIR, "*.txt"))
print(f"Found {len(txt_files)} .txt files")

files_to_process = []
for txt_file in txt_files:
    base_name = os.path.splitext(os.path.basename(txt_file))[0]
    json_file = os.path.join(OUTPUT_DIR, f"{base_name}.json")
    
    if CLOBBER or not os.path.exists(json_file):
        files_to_process.append(txt_file)
    else:
        print(f"Skipping {txt_file} (JSON exists)")

if TEST and files_to_process:
    files_to_process = [files_to_process[0]]
    print(f"TEST mode: processing only {os.path.basename(files_to_process[0])}")

print(f"Will process {len(files_to_process)} files")

Found 158 .txt files
Skipping Multipurpose_Files/portal_files/v1_gsp_num_id_0007.txt (JSON exists)
TEST mode: processing only v1_gsp_num_id_0013.txt
Will process 1 files


In [14]:
# Process files
if not files_to_process or not extractor.pipeline:
    print("Cannot proceed - no files or extractor failed")
else:
    total_triplets = 0
    
    for i, file_path in enumerate(files_to_process, 1):
        file_name = os.path.basename(file_path)
        print(f"\n[{i}/{len(files_to_process)}] Processing {file_name}...")
        
        # Read file
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Split by page breaks
        pages = content.split("<<PAGE_BREAK>>")
        print(f"  Found {len(pages)} pages")
        
        # Get relevant pages for this file
        file_metadata = metadata_df[metadata_df['file_name'] == file_name]
        
        if file_metadata.empty:
            print(f"  No metadata found for {file_name}")
            continue
        
        relevant_pages = set(file_metadata['page_num'].tolist())
        print(f"  Processing {len(relevant_pages)} relevant pages: {sorted(relevant_pages)}")
        
        # Extract triplets from relevant pages
        all_triplets = []
        processed_pages = 0
        
        for page_num in relevant_pages:
            page_index = page_num - 1  # Convert to 0-indexed
            
            if 0 <= page_index < len(pages):
                page_text = pages[page_index].strip()
                
                if page_text:
                    print(f"    Page {page_num}: {len(page_text)} chars")
                    
                    page_triplets = extractor.extract(page_text)
                    
                    # Add metadata to triplets
                    for triplet in page_triplets:
                        triplet['page_num'] = page_num
                        triplet['file_name'] = file_name
                    
                    all_triplets.extend(page_triplets)
                    processed_pages += 1
                    
                    print(f"      → {len(page_triplets)} triplets")
                else:
                    print(f"    Page {page_num}: empty")
            else:
                print(f"    Page {page_num}: not found")
        
        # Save results
        result = {
            "file_name": file_name,
            "processed_pages": processed_pages,
            "total_pages": len(pages),
            "relevant_pages": sorted(relevant_pages),
            "total_triplets": len(all_triplets),
            "triplets": all_triplets,
            "timestamp": pd.Timestamp.now().isoformat()
        }
        
        base_name = os.path.splitext(file_name)[0]
        output_path = os.path.join(OUTPUT_DIR, f"{base_name}.json")
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        
        total_triplets += len(all_triplets)
        print(f"  Saved {len(all_triplets)} triplets to {output_path}")
    
    print(f"\nTotal: {total_triplets} triplets from {len(files_to_process)} files")


[1/1] Processing v1_gsp_num_id_0013.txt...
  Found 642 pages
  Processing 39 relevant pages: [27, 28, 29, 31, 35, 385, 386, 387, 470, 471, 472, 473, 474, 481, 482, 483, 484, 485, 493, 494, 495, 496, 502, 503, 511, 512, 519, 520, 521, 522, 523, 524, 525, 526, 527, 528, 531, 532, 533]
    Page 512: 2535 chars
      → 4 triplets
    Page 385: 4626 chars
      → 7 triplets
    Page 386: 4214 chars
      → 9 triplets
    Page 387: 3883 chars
      → 6 triplets
    Page 519: 4493 chars
      → 5 triplets
    Page 520: 4848 chars
      → 6 triplets
    Page 521: 4673 chars
      → 4 triplets
    Page 522: 4361 chars
      → 5 triplets
    Page 523: 4088 chars
      → 5 triplets
    Page 524: 4401 chars
      → 3 triplets
    Page 525: 4480 chars
      → 6 triplets
    Page 526: 4737 chars
      → 7 triplets
    Page 527: 4313 chars
      → 6 triplets
    Page 528: 2040 chars
      → 4 triplets
    Page 531: 4056 chars


Token indices sequence length is longer than the specified maximum sequence length for this model (1178 > 1024). Running this sequence through the model will result in indexing errors


      → 5 triplets
    Page 532: 2694 chars
      → 2 triplets
    Page 533: 2250 chars
      → 2 triplets
    Page 27: 3831 chars
      → 8 triplets
    Page 28: 4108 chars
      → 7 triplets
    Page 29: 3534 chars
      → 7 triplets
    Page 31: 4480 chars
      → 7 triplets
    Page 35: 3084 chars
      → 3 triplets
    Page 470: 2944 chars
      → 4 triplets
    Page 471: 3459 chars
      → 4 triplets
    Page 472: 4655 chars
      → 6 triplets
    Page 473: 5106 chars
      → 5 triplets
    Page 474: 2920 chars
      → 4 triplets
    Page 481: 4301 chars
      → 6 triplets
    Page 482: 3771 chars
      → 7 triplets
    Page 483: 4524 chars
      → 6 triplets
    Page 484: 5037 chars
      → 6 triplets
    Page 485: 2881 chars
      → 4 triplets
    Page 493: 3485 chars
      → 5 triplets
    Page 494: 4389 chars
      → 8 triplets
    Page 495: 5264 chars
      → 11 triplets
    Page 496: 3717 chars
      → 5 triplets
    Page 502: 3627 chars
      → 6 triplets
    Page 503: 279

In [ ]:
# Analyze results
json_files = glob.glob(os.path.join(OUTPUT_DIR, "*.json"))

if json_files:
    print(f"Analyzing {len(json_files)} result files:")
    
    total_triplets = 0
    total_pages = 0
    all_predicates = []
    
    for json_file in json_files:
        with open(json_file, 'r', encoding='utf-8') as f:
            result = json.load(f)
        
        triplet_count = result.get('total_triplets', 0)
        page_count = result.get('processed_pages', 0)
        
        print(f"  {result['file_name']}: {triplet_count} triplets, {page_count} pages")
        
        total_triplets += triplet_count
        total_pages += page_count
        
        for triplet in result.get('triplets', []):
            all_predicates.append(triplet.get('predicate', ''))
    
    print(f"\nSummary:")
    print(f"  Total triplets: {total_triplets}")
    print(f"  Total pages: {total_pages}")
    print(f"  Avg triplets/page: {total_triplets/total_pages:.1f}" if total_pages > 0 else "")
    
    if all_predicates:
        from collections import Counter
        top_predicates = Counter(all_predicates).most_common(10)
        print(f"\nTop predicates:")
        for pred, count in top_predicates:
            print(f"  {pred}: {count}")
else:
    print("No results found")